In [1]:
import os
import sys

PROJECT_ROOT = "/Users/karima/Ironhack-challenges/fake-news-nlp-classification"

os.chdir(PROJECT_ROOT)

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

print("Current working directory:", os.getcwd())

Current working directory: /Users/karima/Ironhack-challenges/fake-news-nlp-classification


In [2]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.preprocessing.sequence import pad_sequences

from src.preprocessing import preprocess_data

2026-07-10 14:27:59.276211: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
VALIDATION_DATA_PATH = Path("dataset/validation_data.csv")
MODEL_PATH = Path("models/bilstm_model.keras")
TOKENIZER_PATH = Path("models/bilstm_tokenizer.pkl")

OUTPUT_DIR = Path("results/predictions")
OUTPUT_PATH = OUTPUT_DIR / "validation_predictions.csv"

MAX_LENGTH = 300
CLASS_THRESHOLD = 0.5

In [4]:
required_files = {
    "Validation dataset": VALIDATION_DATA_PATH,
    "Bi-LSTM model": MODEL_PATH,
    "Bi-LSTM tokenizer": TOKENIZER_PATH,
}

missing_files = []

for file_name, file_path in required_files.items():
    print(f"{file_name}: {file_path} -> {file_path.exists()}")

    if not file_path.exists():
        missing_files.append(str(file_path))

if missing_files:
    raise FileNotFoundError(
        "The following required files were not found:\n"
        + "\n".join(missing_files)
    )

Validation dataset: dataset/validation_data.csv -> True
Bi-LSTM model: models/bilstm_model.keras -> True
Bi-LSTM tokenizer: models/bilstm_tokenizer.pkl -> True


In [5]:
# 1. Load unseen validation data
validation_data = pd.read_csv(VALIDATION_DATA_PATH)

print("Validation shape:", validation_data.shape)
print("Columns:", validation_data.columns.tolist())

validation_data.head()

Validation shape: (4956, 5)
Columns: ['label', 'title', 'text', 'subject', 'date']


,label,title,text,subject,date
0,2,UK's May 'receiving regular updates' on London...,LONDON (Reuters) - British Prime Minister Ther...,worldnews,"September 15, 2017"
1,2,UK transport police leading investigation of L...,LONDON (Reuters) - British counter-terrorism p...,worldnews,"September 15, 2017"
2,2,Pacific nations crack down on North Korean shi...,WELLINGTON (Reuters) - South Pacific island na...,worldnews,"September 15, 2017"
3,2,Three suspected al Qaeda militants killed in Y...,"ADEN, Yemen (Reuters) - Three suspected al Qae...",worldnews,"September 15, 2017"
4,2,Chinese academics prod Beijing to consider Nor...,BEIJING (Reuters) - Chinese academics are publ...,worldnews,"September 15, 2017"


In [6]:
# 2. Keep a copy of the original validation data
validation_output = validation_data.copy()

In [7]:
# 3. Apply the same preprocessing used during Bi-LSTM training
validation_clean = preprocess_data(validation_data)

print("Cleaned columns:", validation_clean.columns.tolist())

validation_clean[
    ["clean_title", "clean_text", "combined_text"]
].head()

Cleaned columns: ['label', 'title', 'text', 'subject', 'date', 'clean_title', 'clean_text', 'combined_text']


,clean_title,clean_text,combined_text
0,uk may receiving regular update london tube st...,london reuters british prime minister theresa ...,uk may receiving regular update london tube st...
1,uk transport police leading investigation lond...,london reuters british counter terrorism polic...,uk transport police leading investigation lond...
2,pacific nation crack north korean ship fiji pr...,wellington reuters south pacific island nation...,pacific nation crack north korean ship fiji pr...
3,three suspected al qaeda militant killed yemen...,aden yemen reuters three suspected al qaeda mi...,three suspected al qaeda militant killed yemen...
4,chinese academic prod beijing consider north k...,beijing reuters chinese academic publicly broa...,chinese academic prod beijing consider north k...


In [8]:
# 4. Load the fitted tokenizer
tokenizer = joblib.load(TOKENIZER_PATH)

print("Tokenizer loaded successfully.")
print("Vocabulary size:", len(tokenizer.word_index))

Tokenizer loaded successfully.
Vocabulary size: 81240


In [9]:
# 5. Convert text into padded integer sequences
validation_sequences = tokenizer.texts_to_sequences(
    validation_clean["combined_text"]
)

X_validation_padded = pad_sequences(
    validation_sequences,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)

print("Validation padded shape:", X_validation_padded.shape)

Validation padded shape: (4956, 300)


In [10]:
# 6. Load the saved Bi-LSTM model
model = tf.keras.models.load_model(MODEL_PATH)

print("Bi-LSTM model loaded successfully.")
model.summary()

Bi-LSTM model loaded successfully.


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 300, 100)       │     2,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 128)            │        84,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,265,925 (23.90 MB)

 Trainable params: 2,088,641 (7.97 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 4,177,284 (15.94 MB)

In [11]:
# 7. Generate probabilities
prediction_probabilities = model.predict(
    X_validation_padded,
    verbose=1
).ravel()

prediction_probabilities[:10]

155/155 ━━━━━━━━━━━━━━━━━━━━ 5s 31ms/step


array([0.9999587 , 0.9999208 , 0.99996006, 0.99997663, 0.99995077,
       0.99992114, 0.99988693, 0.99987453, 0.99999183, 0.99994475],
      dtype=float32)

In [12]:
# 8. Convert probabilities to labels
predictions = (
    prediction_probabilities >= CLASS_THRESHOLD
).astype(int)

print("Unique predicted labels:", np.unique(predictions))

Unique predicted labels: [0 1]


In [13]:
# 9. Add predictions to the original validation data
validation_output["label"] = predictions
validation_output["prediction_probability"] = prediction_probabilities

validation_output.head()

,label,title,text,subject,date,prediction_probability
0,1,UK's May 'receiving regular updates' on London...,LONDON (Reuters) - British Prime Minister Ther...,worldnews,"September 15, 2017",0.999959
1,1,UK transport police leading investigation of L...,LONDON (Reuters) - British counter-terrorism p...,worldnews,"September 15, 2017",0.999921
2,1,Pacific nations crack down on North Korean shi...,WELLINGTON (Reuters) - South Pacific island na...,worldnews,"September 15, 2017",0.999960
3,1,Three suspected al Qaeda militants killed in Y...,"ADEN, Yemen (Reuters) - Three suspected al Qae...",worldnews,"September 15, 2017",0.999977
4,1,Chinese academics prod Beijing to consider Nor...,BEIJING (Reuters) - Chinese academics are publ...,worldnews,"September 15, 2017",0.999951


In [14]:
# 10. Inspect prediction distribution
print("Prediction counts:")
print(
    validation_output["label"]
    .value_counts()
    .sort_index()
)

print("\nPrediction percentages:")
print(
    validation_output["label"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

Prediction counts:
label
0    3529
1    1427
Name: count, dtype: int64

Prediction percentages:
label
0    71.21
1    28.79
Name: proportion, dtype: float64


In [15]:
# 11. Save predictions
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

validation_output.to_csv(
    OUTPUT_PATH,
    index=False
)

print(f"Validation predictions saved to: {OUTPUT_PATH.resolve()}")

Validation predictions saved to: /Users/karima/Ironhack-challenges/fake-news-nlp-classification/results/predictions/validation_predictions.csv


In [16]:
# 12. Reload and verify the saved file
saved_predictions = pd.read_csv(OUTPUT_PATH)

print("Saved file shape:", saved_predictions.shape)
print("Saved columns:", saved_predictions.columns.tolist())
print("Unique saved labels:", saved_predictions["label"].unique())

saved_predictions.head()

Saved file shape: (4956, 6)
Saved columns: ['label', 'title', 'text', 'subject', 'date', 'prediction_probability']
Unique saved labels: [1 0]


,label,title,text,subject,date,prediction_probability
0,1,UK's May 'receiving regular updates' on London...,LONDON (Reuters) - British Prime Minister Ther...,worldnews,"September 15, 2017",0.999959
1,1,UK transport police leading investigation of L...,LONDON (Reuters) - British counter-terrorism p...,worldnews,"September 15, 2017",0.999921
2,1,Pacific nations crack down on North Korean shi...,WELLINGTON (Reuters) - South Pacific island na...,worldnews,"September 15, 2017",0.999960
3,1,Three suspected al Qaeda militants killed in Y...,"ADEN, Yemen (Reuters) - Three suspected al Qae...",worldnews,"September 15, 2017",0.999977
4,1,Chinese academics prod Beijing to consider Nor...,BEIJING (Reuters) - Chinese academics are publ...,worldnews,"September 15, 2017",0.999951


In [17]:
# 13. Final checks
assert len(saved_predictions) == len(validation_data), (
    "The output must have the same number of rows as the validation data."
)

assert set(saved_predictions["label"].unique()).issubset({0, 1}), (
    "Predicted labels must contain only 0 and 1."
)

assert saved_predictions["prediction_probability"].between(0, 1).all(), (
    "Prediction probabilities must be between 0 and 1."
)

print("All validation checks passed.")

All validation checks passed.


## Final output

The generated file is:

`results/predictions/validation_predictions.csv`

It contains the original validation columns, the predicted `label`,
and the Bi-LSTM `prediction_probability`.